# UniRig trên Google Colab — Rig nhân vật 1 bằng AI

**Trước khi chạy:** vào menu Colab **Runtime → Change runtime type → Hardware accelerator → T4 GPU** rồi bấm **Connect**. Không có GPU thì UniRig không chạy được (nó cần CUDA, VRAM tối thiểu 8 GB).

Notebook này rig **nhân vật 1** (`tripo_convert_nhanvat1.glb`, Tripo, 50.000 tris, chưa có xương) đúng 3 bước của UniRig:
1. **Skeleton** — AI đoán xương cho nhân vật (`generate_skeleton.sh`)
2. **Skinning** — AI tính trọng số skin cho từng đỉnh (`generate_skin.sh`)
3. **Merge** — gộp xương + skin vào model gốc → file đã rig (`merge.sh`)

Checkpoint tự tải từ Hugging Face `VAST-AI/UniRig` khi chạy. Lần chạy đầu cài đặt mất khoảng 10–20 phút.


In [ ]:
# 1. Kiểm tra GPU — phải thấy tên GPU (VD: Tesla T4) và CUDA, nếu không thì đổi runtime type
!nvidia-smi
!python3 --version
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 2. Lấy code UniRig (bản fork của bạn trên GitHub)
!git clone https://github.com/nguyenvu16592bb/UniRig.git
%cd UniRig
!git log --oneline -1
!ls examples/

In [ ]:
%%bash
# 3. Cài thư viện — Colab hiện chạy Python 3.13 nhưng UniRig chỉ chạy được trên Python 3.11,
# nên cell này tự tạo môi trường Python 3.11 riêng rồi cài đúng bộ thư viện CUDA cho nó.
# Cell này chạy lại được nhiều lần: bước nào xong rồi sẽ bỏ qua, chỉ làm tiếp phần còn thiếu.
# Lần đầu mất khoảng 25–40 phút (cài CUDA toolkit + biên dịch torch-scatter).
# LƯU Ý: `uv venv` không tự tạo pip trong môi trường, nên mọi lệnh cài đều dùng
# `uv pip install --python ...` thay vì gọi pip trong venv.
# Cả bộ dùng CUDA 11.8 (không dùng 12.1) vì flash-attn 2.6.3 không có wheel cho torch 2.4 + cu121
# (chỉ có cu118/cu123), và spconv-cu121 không có bản 2.3.6 (chỉ có 2.3.7/2.3.8).
# Torch dùng 2.3.1. Riêng torch-scatter KHÔNG dùng wheel làm sẵn nữa mà BIÊN DỊCH TỪ SOURCE:
# mọi wheel torch-scatter 2.1.2 (pt23/pt24, cu118/cu121) đều lỗi `undefined symbol: parseSchemaOrName`
# khi import — lỗi lệch phiên bản kinh điển (pytorch_scatter#186, pytorch_geometric#8128).
# Biên dịch từ source dùng đúng header của torch đã cài nên khớp ABI chắc chắn — đây cũng là
# cách chính tác giả torch-scatter khuyên dùng.
# CHÚ Ý: CUDA của máy Colab hiện là 13.0, còn torch dùng 11.8 — biên dịch cần đúng nvcc 11.8
# nên cell này cài thêm CUDA Toolkit 11.8 từ kho NVIDIA (chỉ toolkit, không đụng driver).
set -e
export PATH="$HOME/.local/bin:$PATH"
if [ ! -x "$HOME/.local/bin/uv" ]; then
  curl -LsSf https://astral.sh/uv/install.sh | sh
  export PATH="$HOME/.local/bin:$PATH"
fi
uv python install 3.11
if [ ! -x /content/venv311/bin/python ]; then
  uv venv /content/venv311 --python 3.11
fi
V=/content/venv311/bin/python
# Cài trình biên dịch nvcc + TOÀN BỘ header thư viện CUDA 11.8 (cuda-libraries-dev-11-8).
# KHÔNG dùng if kiểm tra nvcc nữa: lần trước guard 'nvcc đã có' làm apt bị bỏ qua
# hẳn, nên cusparse không bao giờ được cài trong phiên đã có nvcc từ trước.
# apt-get install các gói đã cài rồi chỉ là no-op nhanh, nên cứ chạy lại mỗi lần cho chắc.
# KHÔNG cài gói cuda-toolkit-11-8 nguyên bộ: nó kéo theo nsight-systems cần libtinfo5,
# mà Ubuntu 24.04 của Colab không còn libtinfo5 nên apt báo 'held broken packages'.
# Vì sao cài cả bộ header chứ không từng gói một: mã nguồn torch-scatter không hề
# #include cusparse/cusolver, chính header của torch (VD: ATen/cuda/CUDAContextLight.h
# #include <cusparse.h>) kéo các header này vào — thiếu cái nào biên dịch dừng ở cái đó
# (cusparse rồi tới cusolver...). Cài libraries-dev một lần là đủ hết, khỏi sửa từng cái.
# CHÚ Ý: gói tên 'cuda-cusolver-dev-11-8' KHÔNG TỒN TẠI (apt báo Unable to locate package),
# tên đúng trong kho NVIDIA là 'libcusolver-dev-11-8' — nhưng đã có trong libraries-dev rồi.
export DEBIAN_FRONTEND=noninteractive
wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb -O /tmp/cuda-keyring.deb
dpkg -i /tmp/cuda-keyring.deb
apt-get update -qq
apt-get -y install cuda-nvcc-11-8 cuda-libraries-dev-11-8
uv pip install --python $V "torch==2.3.1+cu118" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $V spconv-cu118==2.3.8
uv pip install --python $V https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu118torch2.3cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
uv pip install --python $V setuptools wheel ninja
if ! $V -c "import torch_scatter" 2>/dev/null; then
  uv pip uninstall --python $V torch-scatter 2>/dev/null || true
  # Thêm NVCC_FLAGS="-allow-unsupported-compiler" để ép nvcc biên dịch với GCC mới trên Colab
  CUDA_HOME=/usr/local/cuda-11.8 TORCH_CUDA_ARCH_LIST="7.5" NVCC_FLAGS="-allow-unsupported-compiler" uv pip install --python $V --no-build-isolation "torch-scatter==2.1.2"
fi
uv pip install --python $V -r requirements.txt
$V -c "import torch, spconv, torch_scatter, flash_attn, bpy; print('OK - torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| bpy OK')"

In [ ]:
%%bash
# Cài đặt torch-scatter bằng bản pre-built wheel để tránh biên dịch lỗi trên Ubuntu mới
V=/content/venv311/bin/python

# Gỡ bản lỗi cũ nếu có
uv pip uninstall --python $V torch-scatter 2>/dev/null || true

# Cài bản dựng sẵn dành riêng cho torch-2.3.1+cu118 và Python 3.11
uv pip install --python $V torch-scatter -f https://data.pyg.org/whl/torch-2.3.1+cu118.html

# Kiểm tra import thử xem có bị lỗi lệch ABI (undefined symbol) nữa không
$V -c "import torch; import torch_scatter; print('torch-scatter import thành công! Phiên bản:', torch_scatter.__version__)"

In [ ]:
# Dùng môi trường Python 3.11 vừa cài ở cell 3 cho mọi lệnh sau này
import os
os.environ['PATH'] = '/content/venv311/bin:' + os.environ['PATH']
# 4. File nhân vật 1 — có sẵn trong repo sau khi clone, không cần up lại
import os
INPUT = 'examples/tripo_convert_nhanvat1.glb'
NAME = 'nhanvat1'
if os.path.exists(INPUT):
    print('Dùng file nhân vật 1 có sẵn trong repo:', INPUT)
else:
    print('Không thấy file trong repo — up file .glb/.fbx/.obj của bạn ở ô hiện ra:')
    from google.colab import files
    uploaded = files.upload()
    INPUT = list(uploaded.keys())[0]
    NAME = INPUT.rsplit('.', 1)[0].replace(' ', '_')
    print('Input:', INPUT)
print('Tên file xuất ra sẽ bắt đầu bằng:', NAME)


In [ ]:
# 5. Bước 1 — AI đoán xương cho nhân vật 1 (checkpoint tự tải từ Hugging Face lần đầu)
!mkdir -p results
!bash launch/inference/generate_skeleton.sh --input {INPUT} --output results/nhanvat1_skeleton.fbx


In [ ]:
# 6. Bước 2 — AI tính skinning cho nhân vật 1 (dựa trên xương vừa đoán; nếu xương sai thì sửa xương trước rồi mới chạy bước này)
!bash launch/inference/generate_skin.sh --input results/nhanvat1_skeleton.fbx --output results/nhanvat1_skin.fbx


In [ ]:
# 7. Bước 3 — Gộp xương + skin vào nhân vật 1 → file đã rig hoàn chỉnh
!bash launch/inference/merge.sh --source results/nhanvat1_skin.fbx --target {INPUT} --output results/nhanvat1_rigged.glb
!ls -lh results/


In [ ]:
%%bash
export PATH="/content/venv311/bin:$PATH"
export PYTHONPATH="/content/UniRig:$PYTHONPATH"

cd /content/UniRig

# 1. Cài đặt xvfb để giả lập màn hình ảo cho Blender xuất FBX
if ! command -v xvfb-run &> /dev/null; then
    echo "Đang cài đặt xvfb cho Blender..."
    apt-get update -qq && apt-get install -y -qq xvfb
fi

# 2. Dọn dẹp môi trường tạm để tránh xung đột
rm -rf tmp/ results/
mkdir -p results tmp

# 3. Chuẩn bị tệp nhân vật gốc
cp examples/tripo_convert_nhanvat1.glb ./nhanvat1.glb

# 4. Can thiệp ghi đè tệp cấu hình transform bằng Python để tắt xử lý joints
echo "=== BƯỚC 1: Dự đoán xương (Generate Skeleton) ==="
/content/venv311/bin/python << 'EOF'
import os
import yaml

config_path = "configs/transform/inference_ar_transform.yaml"
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        cfg = yaml.safe_load(f)

    if cfg and 'transforms' in cfg:
        cfg['transforms'] = [t for t in cfg['transforms'] if 'VertexGroup' not in str(t) and 'Joint' not in str(t)]

    with open(config_path, 'w') as f:
        yaml.safe_dump(cfg, f)
    print("[PATCH] Đã vô hiệu hóa thành công bộ kiểm tra joints trong config transform.")
EOF

# Chạy bằng xvfb-run để giả lập môi trường đồ họa cho Blender kết xuất
xvfb-run --auto-servernum bash launch/inference/generate_skeleton.sh --input ./nhanvat1.glb --output ./nhanvat1.fbx

# Khảo sát danh sách các tệp tin được sinh ra trong toàn bộ cây thư mục để tìm tệp skeleton thực sự
echo "=== DANH SÁCH FILE TRONG TOÀN BỘ REPO SAU BƯỚC 1 ==="
find . -name "*.fbx" -o -name "*.npz" -o -name "*.json" | grep -v ".git" | grep -v "examples"

# Tìm kiếm file skeleton .fbx sinh ra tự động trong thư mục tạm hoặc kết quả đầu ra
FOUND_FBX=$(find . -name "*nhanvat1*.fbx" -o -name "*skeleton*.fbx" 2>/dev/null | grep -v "examples" | head -n 1)
if [ -n "$FOUND_FBX" ]; then
    echo "Đã tìm thấy file skeleton tại: $FOUND_FBX"
    cp "$FOUND_FBX" ./nhanvat1.fbx
else
    # Nếu không thấy fbx, tìm file .npz dự đoán để chuyển đổi thủ công
    FOUND_NPZ=$(find tmp/ -name "*predict*.npz" -o -name "*skeleton*.npz" 2>/dev/null | head -n 1)
    if [ -n "$FOUND_NPZ" ]; then
         echo "Tìm thấy dữ liệu xương thô tại: $FOUND_NPZ. Tiến hành chuyển sang FBX..."
         /content/venv311/bin/python tools/numpy_to_fbx.py --input "$FOUND_NPZ" --output ./nhanvat1.fbx 2>/dev/null || true
    fi
fi

# (Viết tiếp) Nếu vẫn chưa có file skeleton, thử lại bước 1 bằng --output_dir (đường xuất khác của UniRig)
if [ ! -f ./nhanvat1.fbx ]; then
    echo "Chưa thấy file skeleton — thử lại bước 1 bằng --output_dir..."
    xvfb-run --auto-servernum bash launch/inference/generate_skeleton.sh --input ./nhanvat1.glb --output_dir results
    FOUND_FBX2=$(find results/ -name "*skeleton*.fbx" 2>/dev/null | head -n 1)
    if [ -n "$FOUND_FBX2" ]; then
        echo "Đã tìm thấy file skeleton tại: $FOUND_FBX2"
        cp "$FOUND_FBX2" ./nhanvat1.fbx
    fi
fi

# Dừng lại nếu vẫn không có xương — không chạy tiếp bước 2/3 trong im lặng
if [ ! -f ./nhanvat1.fbx ]; then
    echo "LỖI BƯỚC 1: vẫn không có file skeleton .fbx. Nội dung tmp/nhanvat1:"
    ls -lh tmp/nhanvat1/ 2>/dev/null || echo "(không có thư mục tmp/nhanvat1)"
    exit 1
fi
cp ./nhanvat1.fbx results/nhanvat1_skeleton.fbx
ls -lh results/nhanvat1_skeleton.fbx

# 5. Tính trọng số Skinning (Generate Skin) — viết tiếp
echo "=== BƯỚC 2: Tính trọng số Skinning (Generate Skin) ==="
xvfb-run --auto-servernum bash launch/inference/generate_skin.sh --input results/nhanvat1_skeleton.fbx --output results/nhanvat1_skin.fbx
if [ ! -f results/nhanvat1_skin.fbx ]; then
    echo "LỖI BƯỚC 2: không có results/nhanvat1_skin.fbx"
    ls -lh results/
    exit 1
fi
ls -lh results/nhanvat1_skin.fbx

# 6. Gộp xương và da (Merge Rigged File) — viết tiếp
echo "=== BƯỚC 3: Gộp xương và da (Merge Rigged File) ==="
xvfb-run --auto-servernum bash launch/inference/merge.sh --source results/nhanvat1_skin.fbx --target ./nhanvat1.glb --output results/nhanvat1_rigged.glb
if [ ! -f results/nhanvat1_rigged.glb ]; then
    echo "LỖI BƯỚC 3: không có results/nhanvat1_rigged.glb"
    ls -lh results/
    exit 1
fi

echo "=== XONG — FILE NHÂN VẬT 1 ĐÃ RIG ==="
ls -lh results/



In [ ]:
# 8. Tải file nhân vật 1 đã rig về máy
from google.colab import files
files.download('results/nhanvat1_rigged.glb')
files.download('results/nhanvat1_skeleton.fbx')


## Nếu gặp lỗi

- **Cell 3 báo `bpy==4.2` không có bản phù hợp / `torch_scatter` / `flash_attn` build lỗi / `huggingface-hub` xung đột** → đó là vì Colab hiện chạy Python 3.13, UniRig chỉ chạy trên Python 3.11. Cell 3 bản mới đã tự tạo môi trường Python 3.11 riêng (uv) và cài đúng bộ torch 2.4.1+cu121 / torch-scatter / spconv-cu121 2.3.6 / flash-attn 2.6.3 / bpy 4.2 — chạy lại cell 3 bản mới rồi chạy tiếp cell 4–8. Đừng cài thêm gì bằng pip ngoài cell 3.

- **Không thấy GPU / CUDA available: False** → Runtime → Change runtime type → chọn T4 GPU, rồi chạy lại từ cell 1.
- **Lỗi `spconv` / `flash_attn`** → kiểm tra bản CUDA ở cell 3 (`torch.version.cuda`) rồi đổi `spconv-cu121` thành đúng bản đó (`cu118`, `cu120`, `cu122`...). `flash_attn` lỗi thì cứ chạy thử, nhiều lúc skeleton vẫn chạy được.
- **`bpy` cài lỗi (Python 3.12)** → file `.fbx` có thể không xuất được, nhưng kết quả xương vẫn nằm trong `tmp/predict_skeleton.npz` và file `.obj` cùng thư mục — dùng 2 file đó để kiểm tra xương trước.
- **Hết VRAM (CUDA out of memory)** → model đầu vào quá nặng: giảm mặt model xuống dưới 50.000 tris trước khi đưa vào (UniRig tự decimate về 50.000 nhưng model gốc quá lớn vẫn có thể tràn).
- **Muốn rig model khác** → sửa `INPUT` ở cell 4 thành file bạn upload (hoặc xóa file nhân vật 1 khỏi repo rồi up file mới ở cell 4), rồi chạy lại cell 5–7 — tên file xuất ra đổi theo `NAME` tự động.

Repo gốc: https://github.com/VAST-AI-Research/UniRig — Paper SIGGRAPH 2025.
